In [15]:
import math
import torch


class LinearLayer:

  def __init__(self, in_features, out_features):
    kaim = math.sqrt(2.0 / in_features)
    self.weights = torch.randn(out_features, in_features) * kaim
    self.bias = torch.zeros(out_features)

    self.weights_grad = torch.zeros_like(self.weights)
    self.bias_grad = torch.zeros_like(self.bias)

  def forward(self, x):
    self.x = x
    return x @ self.weights.T + self.bias

  def backward(self, grad_out):
    grad_inputs = grad_out @ self.weights
    # Assign newly computed gradients
    self.weights_grad = grad_out.T @ self.x
    self.bias_grad = grad_out.sum(dim=0)
    return grad_inputs


class MyRelu:

  def forward(self, x):
    self.x = x
    return torch.clamp(x, min=0)

  def backward(self, grad_out):
    return grad_out * (self.x > 0)


class MyMlp:

  def __init__(self, in_features, hidden_features, out_features):
    self.linear1 = LinearLayer(in_features, hidden_features)
    self.relu = MyRelu()
    self.linear2 = LinearLayer(hidden_features, out_features)
    self.layers = [self.linear1, self.linear2]

  def forward(self, x):
    x = self.linear1.forward(x)
    x = self.relu.forward(x)
    x = self.linear2.forward(x)
    return x

  def backward(self, grad_out):
    grad_linear2 = self.linear2.backward(grad_out)
    grad_relu = self.relu.backward(grad_linear2)
    grad_linear1 = self.linear1.backward(grad_relu)
    return grad_linear1

  def get_params_and_grads(self):
    """Dynamically yields (tensor, grad) pairs with current grad references."""
    for layer in self.layers:
      yield layer.weights, layer.weights_grad
      yield layer.bias, layer.bias_grad


class MyMseLoss:

  def forward(self, y_pred, y_true):
    self.y_pred = y_pred
    self.y_true = y_true
    return torch.mean((y_pred - y_true) ** 2)

  def backward(self):
    n = self.y_pred.numel()
    return (2.0 / n) * (self.y_pred - self.y_true)


class MySgd:

  def __init__(self, model, lr):
    self.model = model
    self.lr = lr

  def zero_grad(self):
    for layer in self.model.layers:
      layer.weights_grad.zero_()
      layer.bias_grad.zero_()

  def step(self):
    for tensor, grad in self.model.get_params_and_grads():
      if grad is not None:
        tensor -= self.lr * grad


# --- Training Loop ---
in_features = 1000
out_features = 10
hidden_features = 128
lr = 0.05

model = MyMlp(in_features, hidden_features, out_features)
loss_fn = MyMseLoss()
optim = MySgd(model, lr)

X = torch.randn(32, in_features)
Y = torch.randn(32, out_features)

for epoch in range(100):
  optim.zero_grad()

  # Forward pass
  y_pred = model.forward(X)

  # Loss computation
  loss = loss_fn.forward(y_pred, Y)

  # Backward pass
  grad_out = loss_fn.backward()
  model.backward(grad_out)

  # Optimization step
  optim.step()

  if epoch % 10 == 0:
    print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f}")

Epoch 00 | Loss: 3.0125
Epoch 10 | Loss: 0.0267
Epoch 20 | Loss: 0.0022
Epoch 30 | Loss: 0.0003
Epoch 40 | Loss: 0.0000
Epoch 50 | Loss: 0.0000
Epoch 60 | Loss: 0.0000
Epoch 70 | Loss: 0.0000
Epoch 80 | Loss: 0.0000
Epoch 90 | Loss: 0.0000
